# Route Resilience — Data Exploration

This notebook explores the SpaceNet Roads dataset, visualises road masks, and baselines
the segmentation model's IoU before and after occlusion augmentation.

**Prerequisites:** Run from `backend/` with the virtual environment active.
```bash
pip install -r requirements.txt
jupyter notebook notebooks/01_data_exploration.ipynb
```

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

DATA_DIR = Path('../data/spacenet_roads')
IMAGE_DIR = DATA_DIR / 'images'
MASK_DIR  = DATA_DIR / 'masks'

print(f'Images: {len(list(IMAGE_DIR.glob("*.png")))} found')
print(f'Masks:  {len(list(MASK_DIR.glob("*.png")))} found')

In [ ]:
# Visualise 6 random image/mask pairs
stems = sorted([p.stem for p in IMAGE_DIR.glob('*.png')])[:6]

fig, axes = plt.subplots(len(stems), 3, figsize=(14, 4 * len(stems)))
fig.patch.set_facecolor('#0B0F1A')

for i, stem in enumerate(stems):
    img = np.array(Image.open(IMAGE_DIR / f'{stem}.png').convert('RGB'))
    mask = np.array(Image.open(MASK_DIR / f'{stem}.png').convert('L')) > 127
    overlay = img.copy()
    overlay[mask] = [0, 229, 180]  # teal road overlay

    for ax, data, title in zip(axes[i], [img, (mask * 255).astype(np.uint8), overlay],
                                 ['Satellite', 'Road Mask', 'Overlay']):
        ax.imshow(data, cmap='gray' if data.ndim == 2 else None)
        ax.set_title(title, color='white', fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('../data/exploration_grid.png', dpi=100, bbox_inches='tight', facecolor='#0B0F1A')
plt.show()
print(f'Road pixel ratio range: {min(m.mean() for m in [np.array(Image.open(MASK_DIR / f"{s}.png").convert("L")) > 127 for s in stems]):.3f} – {max(m.mean() for m in [np.array(Image.open(MASK_DIR / f"{s}.png").convert("L")) > 127 for s in stems]):.3f}')

In [ ]:
# Test occlusion simulation
from app.ml.augmentations import get_train_transforms, get_occluded_test_transforms

stem = stems[0]
img = np.array(Image.open(IMAGE_DIR / f'{stem}.png').convert('RGB'))
mask = (np.array(Image.open(MASK_DIR / f'{stem}.png').convert('L')) > 127).astype(np.float32)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor('#0B0F1A')

occlusion_types = ['canopy', 'shadow', 'cloud', 'vehicle']

for j, occ_type in enumerate(occlusion_types):
    transform = get_occluded_test_transforms(512, occlusion_type=occ_type)
    augmented = transform(image=img, mask=mask)
    aug_img = augmented['image']

    # Denormalize for display
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    if isinstance(aug_img, np.ndarray) and aug_img.max() < 5:
        display_img = np.clip((aug_img.transpose(1,2,0) * std + mean) * 255, 0, 255).astype(np.uint8)
    else:
        display_img = aug_img if isinstance(aug_img, np.ndarray) else np.array(aug_img)

    axes[0, j].imshow(img); axes[0, j].set_title('Original', color='white', fontsize=9); axes[0, j].axis('off')
    axes[1, j].imshow(display_img if display_img.ndim == 3 else display_img, cmap='gray' if display_img.ndim == 2 else None)
    axes[1, j].set_title(f'Occluded: {occ_type}', color='#00E5B4', fontsize=9); axes[1, j].axis('off')

plt.tight_layout()
plt.show()
print('✓ Occlusion simulation transforms verified')

In [ ]:
# Baseline graph construction on a small test mask
from app.graph_pipeline.skeletonize import mask_to_skeleton
from app.graph_pipeline.graph_build import skeleton_to_graph
from app.graph_pipeline.mst_healing import heal_graph
from app.graph_pipeline.metrics import compute_graph_metrics
import networkx as nx

test_mask = (np.array(Image.open(MASK_DIR / f'{stems[0]}.png').convert('L')) > 127).astype(np.uint8)

skel = mask_to_skeleton(test_mask)
G_raw = skeleton_to_graph(skel)
G_healed = heal_graph(G_raw)

raw_m    = compute_graph_metrics(G_raw)
healed_m = compute_graph_metrics(G_healed)

print('Raw graph:')
for k, v in raw_m.items(): print(f'  {k}: {v}')
print('\nHealed graph:')
for k, v in healed_m.items(): print(f'  {k}: {v}')
print(f'\nConnectivity ratio: {healed_m["largest_component_size"] / max(raw_m["largest_component_size"], 1):.3f}')